In [8]:
import pandas as pd
import os

# Load all three raw files
bls = pd.read_csv("../data/raw/bls_unemployment.csv")
fred = pd.read_csv("../data/raw/fred_indicators.csv")
oews = pd.read_csv("../data/raw/oews_wages.csv")

print("BLS shape:", bls.shape)
print("FRED shape:", fred.shape)
print("OEWS shape:", oews.shape)

BLS shape: (108, 6)
FRED shape: (144, 3)
OEWS shape: (1403, 6)


In [9]:
print("=== BLS ===")
print(bls.head(3))
print(bls.dtypes)
print()
print("=== FRED ===")
print(fred.head(3))
print(fred.dtypes)
print()
print("=== OEWS ===")
print(oews.head(3))
print(oews.dtypes)

=== BLS ===
     series_id industry  year period     month  unemployment_rate
0  LNS14000000    total  2024    M12  December                4.1
1  LNS14000000    total  2024    M11  November                4.2
2  LNS14000000    total  2024    M10   October                4.1
series_id                str
industry                 str
year                   int64
period                   str
month                    str
unemployment_rate    float64
dtype: object

=== FRED ===
         series_name        date  value
0  unemployment_rate  2022-01-01    4.0
1  unemployment_rate  2022-02-01    3.9
2  unemployment_rate  2022-03-01    3.7
series_name        str
date               str
value          float64
dtype: object

=== OEWS ===
  OCC_CODE               OCC_TITLE O_GROUP    TOT_EMP H_MEDIAN A_MEDIAN
0  00-0000         All Occupations   total  154187380     23.8    49500
1  11-0000  Management Occupations   major   10966830     58.7   122090
2  11-1000          Top Executives   minor    382

In [10]:
# Create a proper date column in BLS by combining year and month
bls["date"] = pd.to_datetime(
    bls["year"].astype(str) + "-" + bls["month"] + "-01",
    format="%Y-%B-%d"
)

# Drop the columns we no longer need
bls = bls.drop(columns=["year", "period", "month"])

print(bls.head(3))
print(bls.dtypes)

     series_id industry  unemployment_rate       date
0  LNS14000000    total                4.1 2024-12-01
1  LNS14000000    total                4.2 2024-11-01
2  LNS14000000    total                4.1 2024-10-01
series_id                       str
industry                        str
unemployment_rate           float64
date                 datetime64[us]
dtype: object


In [11]:
# Reload fred now that it has all four series
fred = pd.read_csv("../data/raw/fred_indicators.csv")

# Pivot FRED so each indicator becomes its own column
fred["date"] = pd.to_datetime(fred["date"])
fred["value"] = pd.to_numeric(fred["value"], errors="coerce")

fred_pivot = fred.pivot_table(
    index="date",
    columns="series_name",
    values="value"
).reset_index()

fred_pivot.columns.name = None
print(fred_pivot.head(3))
print(fred_pivot.shape)

        date  JOLTS_openings  avg_hourly_earnings      cpi  unemployment_rate
0 2022-01-01         11210.0                31.60  282.543                4.0
1 2022-02-01         11698.0                31.64  284.500                3.9
2 2022-03-01         12301.0                31.80  287.674                3.7
(36, 5)


In [12]:
# Join BLS unemployment and FRED indicators on date
combined = pd.merge(bls, fred_pivot, on="date", how="left")

print(combined.shape)
print(combined.head(3))

(108, 8)
     series_id industry  unemployment_rate_x       date  JOLTS_openings  \
0  LNS14000000    total                  4.1 2024-12-01          7295.0   
1  LNS14000000    total                  4.2 2024-11-01          7566.0   
2  LNS14000000    total                  4.1 2024-10-01          7379.0   

   avg_hourly_earnings      cpi  unemployment_rate_y  
0                35.69  317.604                  4.1  
1                35.60  316.528                  4.2  
2                35.46  315.631                  4.1  


In [14]:
print(combined.columns.tolist())

['series_id', 'industry', 'unemployment_rate_x', 'date', 'JOLTS_openings', 'avg_hourly_earnings', 'cpi', 'unemployment_rate_y']


In [15]:
# Drop the FRED unemployment rate since we have it from BLS already
combined = combined.drop(columns=["unemployment_rate_y"])

# Rename the BLS one back to something clear
combined = combined.rename(columns={"unemployment_rate_x": "unemployment_rate_bls"})

print(combined.columns.tolist())

['series_id', 'industry', 'unemployment_rate_bls', 'date', 'JOLTS_openings', 'avg_hourly_earnings', 'cpi']


In [16]:
# Sort by industry and date so calculations are in the right order
combined = combined.sort_values(["industry", "date"])

# Year over year change in unemployment per industry
combined["unemployment_yoy_change"] = combined.groupby("industry")[
    "unemployment_rate_bls"
].pct_change(periods=12) * 100

# 3 month rolling average of unemployment per industry
combined["unemployment_3m_avg"] = combined.groupby("industry")[
    "unemployment_rate_bls"
].transform(lambda x: x.rolling(3).mean())

# Wage growth vs inflation ratio
combined["wage_vs_inflation"] = (
    combined["avg_hourly_earnings"] / combined["cpi"]
)

print(combined.columns.tolist())
print(combined.shape)
print(combined.head(3))

['series_id', 'industry', 'unemployment_rate_bls', 'date', 'JOLTS_openings', 'avg_hourly_earnings', 'cpi', 'unemployment_yoy_change', 'unemployment_3m_avg', 'wage_vs_inflation']
(108, 10)
      series_id      industry  unemployment_rate_bls       date  \
71  LNU04032231  construction                    7.1 2022-01-01   
70  LNU04032231  construction                    6.7 2022-02-01   
69  LNU04032231  construction                    6.0 2022-03-01   

    JOLTS_openings  avg_hourly_earnings      cpi  unemployment_yoy_change  \
71         11210.0                31.60  282.543                      NaN   
70         11698.0                31.64  284.500                      NaN   
69         12301.0                31.80  287.674                      NaN   

    unemployment_3m_avg  wage_vs_inflation  
71                  NaN           0.111841  
70                  NaN           0.111213  
69                  6.6           0.110542  


In [17]:
output_path = "../data/processed/combined_labor_data.csv"
combined.to_csv(output_path, index=False)
print(f"Saved! Shape: {combined.shape}")
print(f"Columns: {combined.columns.tolist()}")

Saved! Shape: (108, 10)
Columns: ['series_id', 'industry', 'unemployment_rate_bls', 'date', 'JOLTS_openings', 'avg_hourly_earnings', 'cpi', 'unemployment_yoy_change', 'unemployment_3m_avg', 'wage_vs_inflation']
